In [0]:
import time
import json
# Create a directory for streaming data
dbutils.fs.mkdirs("/Volumes/workspace/default/streaming_data")
# Simulate writing JSON events
for i in range(5):
 event = {"event_id": i, "sensor_value": i*10, "timestamp": str(time.time())}
 path = f"/Volumes/workspace/default/streaming_data/event_{i}.json"
 dbutils.fs.put(path, json.dumps(event), overwrite=True)

Wrote 69 bytes.
Wrote 70 bytes.
Wrote 70 bytes.
Wrote 70 bytes.
Wrote 70 bytes.


In [0]:
# Read streaming data
stream_df = (
 spark.readStream
 .format("json")
 .schema("event_id INT, sensor_value INT, timestamp STRING")
 .load("/Volumes/workspace/default/streaming_data")
)
# Display stream in real-time
display() 

In [0]:
# Aggregate sensor values
agg_df = (
 stream_df.groupBy("event_id")
 .agg({"sensor_value": "avg"})
 .withColumnRenamed("avg(sensor_value)", "avg_value")
)
# Write results to memory table
query = (
 agg_df.writeStream
 .outputMode("complete")
 .format("memory")
 .queryName("sensor_avg")
 .trigger(availableNow=True)
 .option("checkpointLocation", "/Volumes/workspace/default/streaming_data/checkpoint_agg")
 .start()
)


In [0]:
%sql
-- Run this in a SQL cell
SELECT * FROM sensor_avg;

event_id,avg_value
1,10.0
3,30.0
4,40.0
2,20.0
0,0.0


Databricks visualization. Run in Databricks to view.